<a href="https://colab.research.google.com/github/sweet-cross/cross_notebooks/blob/main/notebooks/workshop_20260309_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
if 'google.colab' in sys.modules:
    # Installs the repo as a package, along with its dependencies
    !pip install git+https://github.com/sweet-cross/cross_notebooks.git

# CROSS Results

This notebook uses the [CrossContracts](https://sweet-cross.github.io/crosscontract/)
package and the *CrossClient* therein to interact with assumptions and the preliminary 
results for the CROSS 2025 scenarios.

To be able to interact with the assumptions using the CrossClient you need an 
account on the Cross platform. You can create an account here: (Register Account)[https://sweet-cross.ch/register]

Results available are given under the following contracts:
- result_electricity_consumption
- result_electricity_supply
- result_h2_fec
- result_h2_supply
- result_liquids_consumption
- result_liquids_supply
- result_methane_consumption
- result_methane_supply
- result_process_heat_energy_production
- result_space_heat_energy_supply
- result_district_heat_energy_production
- result_passenger_road_public_fec
- result_passenger_road_private_fec
- result_freight_road_fec
- result_storage_installed_volume
- result_storage_output
- result_elec_cons_typical_day
- result_elec_supply_typical_day

Assumptions available are:
- scenass_aviation_fuel_demand
- scenass_biomass_potential
- scenass_cost_generation_technologies
- scenass_cost_heating_technologies
- scenass_cost_hydrogen_technologies
- scenass_cost_storage_technologies
- scenass_electric_appliances_useful_energy_demand
- scenass_energy_reference_area_totals
- scenass_energy_reference_area
- scenass_freight_transport_useful_energy_demand
- scenass_gdp
- scenass_hdd_cross
- scenass_households
- scenass_import_prices
- scenass_passenger_transport_useful_energy_demand
- scenass_population
- scenass_process_heat_useful_energy_demand
- scenass_space_heating_useful_energy_demand
- scenass_warm_water_useful_energy_demand
- scenass_working_population

An overview is provided on the [data model page](https://sweet-cross.github.io/data-model/docs/variables/results/).

## Packages and options

In [2]:
from crosscontract import CrossClient
import getpass
import pandas as pd

## Connecting to CROSS

To connect to CROSS and to use the API, you use the CrossClient given your username
and password. You can either use your username or mail address to connect.

In [3]:
username = input("Enter your username or mail address: ")
password = getpass.getpass("Enter your password: ")
my_client = CrossClient(username=username, password=password)

## Get data


In [4]:
def get_data(
    contract_name: str, 
    scenario_name: str | None = None,
    scenario_variant: str | None = None,
    model: str | None = None,
    client: CrossClient = my_client
) -> pd.DataFrame:
    """Get CROSS 2025 data for a specified contract.
    
    Args:
        contract_name (str): The name of the contract to retrieve results for.
        scenario_name (str | None, optional): The name of the scenario to filter results by.
            Defaults to None, which means no filtering by scenario name.
        scenario_variant (str | None, optional): The name of the scenario variant to filter results by.
            Defaults to None, which means no filtering by scenario variant.
        model (str | None, optional): The name of the model to filter results by.
            Defaults to None, which means no filtering by model.
            Only valid in case of a result contract.
        client (CrossClient, optional): An instance of CrossClient to use for 
            retrieving data. 
            Defaults to my_client.
            
    Returns:
        pd.DataFrame: A DataFrame containing the results for the specified contract.
    """
    filters = {
        "scenario_group": "cross202506"
    }
    if scenario_name is not None:
        filters["scenario_name"] = scenario_name
    if scenario_variant is not None:
        filters["scenario_variant"] = scenario_variant
    if contract_name.startswith("result_") and model is not None:
        filters["model"] = model
    df = (
        client.contracts.get(contract_name).get_data(filters=filters)
        .drop(columns=["scenario_group"])
    )
    return df

In [5]:
df_elec_cons = get_data("result_electricity_consumption", scenario_name="abroad-res-full")